# Feature Engineering for Image Steganalysis

## 1. Statistical Feature Extraction

This notebook extracts statistical features from RGB cover
and stego images.

The extracted features will be used to train and compare
Support Vector Machine and Random Forest classifiers.

Features:
- LSB one ratio
- LSB neighbor agreement
- Even-odd histogram imbalance
- LSB and second bit-plane agreement

The first 112 RGB channel positions are excluded
to avoid using the fixed V2 protocol header.

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from PIL import Image
import numpy as np
import csv

# Locate the project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

manifest_path = project_root / "dataset" / "ucid_manifest.csv"
output_path = project_root / "dataset" / "ucid_features.csv"

HEADER_BITS = 112

def extract_image_features(image_path):
    """Extract 12 statistical features from an RGB image."""

    with Image.open(image_path) as image:
        rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)

    values = rgb.reshape(-1)
    features = {}

    for channel_index, channel_name in enumerate(["r", "g", "b"]):

        # Skip all header positions in the interleaved RGB stream
        start = HEADER_BITS + ((channel_index - HEADER_BITS) % 3)
        channel = values[start::3]

        if len(channel) < 2:
            raise ValueError("Image is too small.")

        lsb = channel & 1
        second_bit = (channel >> 1) & 1

        # Feature 1: Proportion of one bits
        features[f"{channel_name}_lsb_one_ratio"] = float(
            np.mean(lsb)
        )

        # Feature 2: Agreement between neighboring LSBs
        features[f"{channel_name}_lsb_neighbor_agreement"] = float(
            np.mean(lsb[:-1] == lsb[1:])
        )

        # Feature 3: Even-odd histogram pair imbalance
        histogram = np.bincount(channel, minlength=256)

        differences = np.abs(
            histogram[::2].astype(np.int64)
            - histogram[1::2].astype(np.int64)
        )

        features[f"{channel_name}_histogram_imbalance"] = float(
            differences.sum() / histogram.sum()
        )

        # Feature 4: Agreement between bit planes
        features[f"{channel_name}_bitplane_agreement"] = float(
            np.mean(lsb == second_bit)
        )

    return features

def extract_advanced_features(image_path):
    """Extract spatial LSB features from an RGB image."""

    with Image.open(image_path) as image:
        rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)

    height, width, _ = rgb.shape

    # Exclude the first 112 RGB channel positions
    valid_mask = (
        np.arange(height * width * 3)
        .reshape(height, width, 3) >= 112
    )

    features = {}

    for channel_index, name in enumerate(["r", "g", "b"]):

        lsb = rgb[:, :, channel_index] & 1
        valid = valid_mask[:, :, channel_index]

        # Horizontal neighboring pixels
        horizontal_valid = valid[:, :-1] & valid[:, 1:]

        horizontal_equal = (
            lsb[:, :-1] == lsb[:, 1:]
        )

        features[f"{name}_horizontal_agreement"] = float(
            np.mean(horizontal_equal[horizontal_valid])
        )

        # Vertical neighboring pixels
        vertical_valid = valid[:-1, :] & valid[1:, :]

        vertical_equal = (
            lsb[:-1, :] == lsb[1:, :]
        )

        features[f"{name}_vertical_agreement"] = float(
            np.mean(vertical_equal[vertical_valid])
        )

    return features


def extract_image_features_v2(image_path):
    """Combine basic and advanced statistical features."""

    basic = extract_image_features(image_path)
    advanced = extract_advanced_features(image_path)

    return {**basic, **advanced}

# Read the labeled dataset
with open(manifest_path, "r", encoding="utf-8", newline="") as file:
    records = list(csv.DictReader(file))

# Start with 20 images, then process the complete dataset
MAX_SAMPLES = None

selected_records = (
    records if MAX_SAMPLES is None
    else records[:MAX_SAMPLES]
)

feature_records = []

for record in selected_records:

    image_path = project_root / record["file"]
    features = extract_image_features_v2(image_path)

    feature_records.append({
        "source_id": record["source_id"],
        "split": record["split"],
        "label": int(record["label"]),
        "method": record["method"],
        "payload_rate": record["payload_rate"],
        **features
    })

# Save extracted features
with open(output_path, "w", encoding="utf-8", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=list(feature_records[0].keys())
    )

    writer.writeheader()
    writer.writerows(feature_records)

print("Images processed:", len(feature_records))
print("Features per image:", len(features))
print("Feature file:", output_path)

for name, value in features.items():
    print(f"{name}: {value:.6f}")

Images processed: 2100
Features per image: 18
Feature file: C:\Users\hp\Projects\image-steganography\dataset\ucid_features.csv
r_lsb_one_ratio: 0.489511
r_lsb_neighbor_agreement: 0.499916
r_histogram_imbalance: 0.044093
r_bitplane_agreement: 0.498748
g_lsb_one_ratio: 0.489275
g_lsb_neighbor_agreement: 0.500519
g_histogram_imbalance: 0.038184
g_bitplane_agreement: 0.501931
b_lsb_one_ratio: 0.489901
b_lsb_neighbor_agreement: 0.500290
b_histogram_imbalance: 0.035558
b_bitplane_agreement: 0.502298
r_horizontal_agreement: 0.499939
r_vertical_agreement: 0.497302
g_horizontal_agreement: 0.500605
g_vertical_agreement: 0.499916
b_horizontal_agreement: 0.500207
b_vertical_agreement: 0.502705


## 2. Feature Dataset Validation

This section verifies the number of samples, class labels,
statistical features, and dataset splits before model training.

In [3]:
import csv
import math
from collections import Counter, defaultdict

# Load extracted features
with open(output_path, "r", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

# Identify the 12 numerical features
feature_columns = list(extract_image_features_v2(
    project_root / records[0]["file"]
).keys())

# Check dataset dimensions
assert len(rows) == 2100, "Unexpected sample count."
assert len(feature_columns) == 18, "Unexpected feature count."

# Check that every numerical feature is valid
for row in rows:
    for feature in feature_columns:
        assert math.isfinite(float(row[feature]))

# Check class distribution
labels = Counter(row["label"] for row in rows)

assert labels["0"] == 300
assert labels["1"] == 1800

# Check dataset splits
splits = Counter(row["split"] for row in rows)

# Check that each source image belongs to only one split
source_splits = defaultdict(set)

for row in rows:
    source_splits[row["source_id"]].add(row["split"])

assert all(len(value) == 1 for value in source_splits.values())

print("Total samples:", len(rows))
print("Features per image:", len(feature_columns))
print("Labels:", dict(labels))
print("Splits:", dict(splits))
print("Feature dataset validation passed.")

Total samples: 2100
Features per image: 18
Labels: {'0': 300, '1': 1800}
Splits: {'train': 1470, 'validation': 315, 'test': 315}
Feature dataset validation passed.


## 3. Residual-Based Feature Engineering

This experiment extends the existing 18 statistical
features with six residual-based features.

A four-neighbor predictor estimates pixel values.

The features measure:
- Mean absolute prediction residual
- Proportion of near-zero residuals

The features are computed separately for RGB channels.
Header-affected pixel neighborhoods are excluded.

The new dataset is saved separately for comparison
with the previous feature sets.

In [4]:
def extract_residual_features(image_path):
    """Extract six residual-based features."""

    with Image.open(image_path) as image:
        rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)

    height, width, _ = rgb.shape

    valid_mask = (
        np.arange(height * width * 3)
        .reshape(height, width, 3) >= 112
    )

    features = {}

    for c, name in enumerate(["r", "g", "b"]):

        channel = rgb[:, :, c].astype(np.float32)
        valid = valid_mask[:, :, c]

        # Predict the center pixel from four neighbors
        prediction = (
            channel[:-2, 1:-1]
            + channel[2:, 1:-1]
            + channel[1:-1, :-2]
            + channel[1:-1, 2:]
        ) / 4.0

        center = channel[1:-1, 1:-1]

        # Exclude pixels affected by the V2 header
        valid_region = (
            valid[1:-1, 1:-1]
            & valid[:-2, 1:-1]
            & valid[2:, 1:-1]
            & valid[1:-1, :-2]
            & valid[1:-1, 2:]
        )

        residual = np.abs(
            center - prediction
        )[valid_region]

        if residual.size == 0:
            raise ValueError("Image is too small.")

        features[f"{name}_residual_mae"] = float(
            np.mean(residual) / 255.0
        )

        features[f"{name}_residual_near_zero_ratio"] = float(
            np.mean(residual <= 1.0)
        )

    return features


def extract_image_features_v3(image_path):
    """Combine 18 existing features with six residual features."""

    previous = extract_image_features_v2(image_path)
    residual = extract_residual_features(image_path)

    return {**previous, **residual}

In [5]:
from pathlib import Path
import csv

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

manifest_path = project_root / "dataset" / "ucid_manifest.csv"
new_output_path = project_root / "dataset" / "ucid_features_v3.csv"

with open(manifest_path, "r", encoding="utf-8", newline="") as file:
    dataset_records = list(csv.DictReader(file))

new_feature_records = []

for record in dataset_records:

    image_path = project_root / record["file"]

    features = extract_image_features_v3(image_path)

    new_feature_records.append({
        "source_id": record["source_id"],
        "split": record["split"],
        "label": int(record["label"]),
        "method": record["method"],
        "payload_rate": record["payload_rate"],
        **features
    })

with open(new_output_path, "w", encoding="utf-8", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=list(new_feature_records[0].keys())
    )

    writer.writeheader()
    writer.writerows(new_feature_records)

print("Images processed:", len(new_feature_records))
print("Features per image:", len(features))
print("Saved:", new_output_path)

assert len(new_feature_records) == 2100
assert len(features) == 24

print("Residual feature dataset validation passed.")

Images processed: 2100
Features per image: 24
Saved: C:\Users\hp\Projects\image-steganography\dataset\ucid_features_v3.csv
Residual feature dataset validation passed.
